# E791 $D^+\to\pi^-\pi^+\pi^+$ — $\rho(1450)$ mass/width closure with uniform Dalitz MC normalization

This is the Monte Carlo counterpart of notebook 07. The only normalization change is `DalitzGrid` $\to$ `DalitzMC`: random points uniformly distributed in physical Dalitz area with constant integration weights.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from dalitzplotfitter import (DalitzMC,DecayChannel,DecayModel,Minimizer,NonResonant,Parameter,RealImag,Resonance,enable_x64,weighted_resample)
enable_x64()


In [ ]:
channel=DecayChannel("D+",("pi-","pi+","pi+"))
fit2_polar={"sigma":(1.17,205.7),"rho770":(1.00,0.0),"NR":(0.48,57.3),"f0_980":(0.43,165.0),"f2_1270":(0.76,57.3),"f0_1370":(0.26,105.4),"rho1450":(0.14,319.1)}
MTRUE=1.4650; WTRUE=0.3100
def polar_to_xy(r,p): p=np.deg2rad(p); return r*np.cos(p),r*np.sin(p)
def internal_xy(n):
    r,p=fit2_polar[n]
    if n=="NR": p+=180.0
    return polar_to_xy(r,p)
truth_xy={n:internal_xy(n) for n in fit2_polar}; truth={}
def free_coeff(n):
    x,y=truth_xy[n]; truth[f"{n}.x"]=float(x); truth[f"{n}.y"]=float(y)
    return RealImag(Parameter.coefficient(f"{n}.x",0.0,owner=n,step=0.01),Parameter.coefficient(f"{n}.y",0.0,owner=n,step=0.01))
c={"sigma":free_coeff("sigma"),"rho770":RealImag(1.0,0.0),"NR":free_coeff("NR"),"f0_980":free_coeff("f0_980"),"f2_1270":free_coeff("f2_1270"),"f0_1370":free_coeff("f0_1370"),"rho1450":free_coeff("rho1450")}
mpar=Parameter.dynamics("rho1450.mass",MTRUE,owner="rho1450",backend_name="pole_mass",bounds=(1.20,1.75),step=0.002)
wpar=Parameter.dynamics("rho1450.width",WTRUE,owner="rho1450",backend_name="pole_width",bounds=(0.05,0.70),step=0.002)
truth["rho1450.mass"]=MTRUE; truth["rho1450.width"]=WTRUE
model=DecayModel(channel,[
Resonance("sigma",(0,1),c["sigma"],mass=0.4780,width=0.3240,spin=0,resonance_radius=3.0,parent_radius=3.0),
Resonance("rho770",(0,1),c["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
Resonance("f0_980",(0,1),c["f0_980"],mass=0.9750,width=0.0440,spin=0,resonance_radius=3.0,parent_radius=3.0),
Resonance("f2_1270",(0,1),c["f2_1270"],mass=1.2750,width=0.1850,spin=2,resonance_radius=3.0,parent_radius=3.0),
Resonance("f0_1370",(0,1),c["f0_1370"],mass=1.4340,width=0.1730,spin=0,resonance_radius=3.0,parent_radius=3.0),
Resonance("rho1450",(0,1),c["rho1450"],mass=mpar,width=wpar,spin=1,resonance_radius=3.0,parent_radius=3.0),
NonResonant(c["NR"]),])


## Uniform Dalitz MC normalization


In [ ]:
N_NORM=1_000_000
norm=DalitzMC(channel.parent_mass,channel.daughter_masses).generate(N_NORM,seed=2027)
print("normalization points =",norm.size)
print("constant weights =",bool(jnp.all(norm.weights==norm.weights[0])))


## Generate toy and fit


In [ ]:
pool=model.generate_phase_space(1_000_000,seed=2000)
truth_cache=model.prepare_cache(pool,norm)
truth_intensity,_=truth_cache.evaluate(truth)
data=weighted_resample(jax.random.key(791),pool,pool.weights*truth_intensity,100_000,replace=True)
cache=model.prepare_cache(data,norm)
def nll(values):
    intensity,normalization=cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity,min=1e-300)))+data.size*jnp.log(normalization)
minimizer=Minimizer(nll,model.parameters,tolerance=1e-4,verbose=2)
rng=np.random.default_rng(314159); start_values={}
for p in model.parameters:
    if p.fixed: continue
    if p.kind.value=="coefficient": start_values[p.name]=float(rng.uniform(-2.5,2.5))
    elif p.name=="rho1450.mass": start_values[p.name]=float(rng.uniform(1.30,1.65))
    elif p.name=="rho1450.width": start_values[p.name]=float(rng.uniform(0.15,0.50))
print("NLL(truth)=",float(nll(truth)))
print("NLL(start)=",float(nll(start_values)))


In [ ]:
gradient_check=minimizer.check_gradient(start_values,step_scale=1e-5,print_table=True)


In [ ]:
result=minimizer.fit(start_values=start_values,simplex=False,ncall=100000)
print("valid=",bool(result.valid))
print("NLL(fit)=",float(result.fval))
print("NLL(truth)=",float(nll(truth)))
print("fit-truth NLL=",float(result.fval-nll(truth)))
print("EDM=",float(result.fmin.edm))
print(f"{'parameter':18s} {'truth':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed: continue
    t=float(truth[p.name]); f=float(result.values[p.name]); e=float(result.errors[p.name]); pull=(f-t)/e
    print(f"{p.name:18s} {t:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")
